# Atlantic zonal-mean temperature and salinity biases

Latitude–depth sections of the Atlantic zonal-mean potential temperature and salinity bias, for ACCESS-OM3 (relative to WOA23) and ACCESS-OM2 `025deg_jra55_iaf_omip2_cycle1` (relative to WOA13), plus the WOA23 climatology.

Issue: [#195 AMOC bias: zonal mean lat/depth section of biases in T/S in the Atlantic](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/195). This is the Atlantic-only counterpart of the global figure in `temp-salt-vs-depth-latitude.ipynb` (Kiss et al. 2020, Fig. 12).

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name

In [ ]:
if not papermill:
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import numpy as np
import xarray as xr
import intake
import cmocean
import matplotlib.pyplot as plt
from scipy import ndimage
from distributed import Client
from exptdata_access import get_exptname_from_path

client = Client(threads_per_worker=1)
om3exptname = get_exptname_from_path(esm_file)
om2exptname = "025deg_jra55_iaf_omip2_cycle1"

# Common averaging period: the last 25 years both runs cover (OM2 IAF cycle 1 ends in 2018)
period = slice("1994", "2018")

## Atlantic basin mask

Ocean points connected to the mid-Atlantic (0°N, 25°W) within 100°W–25°E, 34°S–65°N. Narrow cuts at the Strait of Gibraltar, Hudson Strait and the Danish straits exclude the Mediterranean, Hudson Bay and the Baltic; the Gulf of Mexico and Caribbean are included. The 65°N limit keeps us south of the tripolar region, so the nominal 1D coordinates are true longitude/latitude. On the ACCESS-OM2 grid this agrees with the CMIP-style basin mask (`/g/data/ik11/grids/om2_p25deg_basinmask_yt_xt.nc`, code 2) at 99.5% of points.

In [ ]:
def atlantic_mask(wet):
    """Boolean Atlantic mask from a 2D (lat, lon) wet mask with 1D nominal lon/lat coords."""
    lat, lon = xr.broadcast(wet[wet.dims[0]], wet[wet.dims[1]])
    box = wet & (lat >= -34) & (lat <= 65) & (lon >= -100) & (lon <= 25)
    cuts = (((lon > -6) & (lon < -5) & (lat > 34) & (lat < 38))       # Gibraltar
            | ((lon > -71) & (lon < -69) & (lat > 58))                # Hudson Strait
            | ((lon > 9.5) & (lon < 10.5) & (lat > 53) & (lat < 60)))  # Danish straits
    labels, _ = ndimage.label((box & ~cuts).values)
    seed = labels[np.abs(lat[:, 0] - 0).argmin().item(), np.abs(lon[0] + 25).argmin().item()]
    return wet.copy(data=labels == seed)

## Load ACCESS-OM3 potential temperature (`thetao`) and salinity (`so`)

In [ ]:
om3cat = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=["variable", "variable_long_name", "variable_standard_name",
                            "variable_cell_methods", "variable_units"],
)

def load_om3(variable):
    return om3cat.search(variable=variable, frequency="1mon").to_dask(
        xarray_open_kwargs=dict(chunks={"time": 1}, decode_timedelta=True),
        xarray_combine_by_coords_kwargs=dict(compat="override", data_vars="minimal", coords="minimal"),
    )[variable]

om3 = {"T": load_om3("thetao"), "S": load_om3("so")}
om3atl = atlantic_mask(om3["T"].isel(time=0, z_l=0).notnull().load())

## Load ACCESS-OM2 potential temperature (`pot_temp`) and salinity (`salt`)

In [ ]:
om2cat = intake.cat.access_nri[om2exptname]

def load_om2(variable):
    return om2cat.search(variable=variable, frequency="1mon").to_dask(
        xarray_open_kwargs=dict(chunks={"time": 1}, decode_timedelta=True),
        xarray_combine_by_coords_kwargs=dict(compat="override", data_vars="minimal", coords="minimal"),
    )[variable]

om2 = {"T": load_om2("pot_temp") - 273.15, "S": load_om2("salt")}  # K -> degC
om2atl = atlantic_mask(om2["S"].isel(time=0, st_ocean=0).notnull().load())

Check the masks on both grids:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
om3atl.sel(xh=slice(-110, 40)).plot(ax=axs[0], add_colorbar=False)
om2atl.sel(xt_ocean=slice(-110, 40)).plot(ax=axs[1], add_colorbar=False)
axs[0].set_title("ACCESS-OM3 Atlantic mask")
axs[1].set_title("ACCESS-OM2 Atlantic mask")
plt.show()

## Time-mean Atlantic zonal means

Mean over the monthly output in `period`, then the mean over Atlantic points along each latitude row (grid spacing is uniform along a row south of 65°N).

In [ ]:
def atlantic_zonal_mean(da, mask):
    ydim, xdim = mask.dims
    rows, cols = mask.any(xdim).values.nonzero()[0], mask.any(ydim).values.nonzero()[0]
    subset = {ydim: slice(rows[0], rows[-1] + 1), xdim: slice(cols[0], cols[-1] + 1)}
    m = mask.isel(subset)
    return da.isel(subset).where(m).mean(xdim)  # only read the Atlantic's bounding box

om3_zm = {v: atlantic_zonal_mean(da.sel(time=period).mean("time"), om3atl).load() for v, da in om3.items()}
om2_zm = {v: atlantic_zonal_mean(da.sel(time=period).mean("time"), om2atl).load() for v, da in om2.items()}

## Observations

WOA23 and WOA13 monthly climatologies already interpolated onto the OM3 and OM2 grids (as in `temp-salt-vs-depth-latitude.ipynb`), averaged over the 12 months and masked to each model's Atlantic wet points so model and observations are averaged over exactly the same points.

In [ ]:
woa23 = (xr.open_mfdataset("/g/data/ik11/inputs/access-om3/woa23/025/2026.06.11/woa23_ts_*_mom.nc")
         .mean("time").rename({"lon": "xh", "lat": "yh", "depth": "z_l"}))
woa23 = woa23.assign_coords(xh=om3atl.xh, yh=om3atl.yh, z_l=om3["T"].z_l)  # same grid as OM3
woa23_zm = {v: atlantic_zonal_mean(woa23[name].where(om3[v].isel(time=0).notnull()), om3atl).load()
            for v, name in [("T", "ptemp"), ("S", "salt")]}

woa13 = (xr.open_mfdataset("/g/data/ik11/observations/woa13/025_KDS50/woa13_ts_??_mom025.nc",
                           decode_times=xr.coders.CFDatetimeCoder(use_cftime=True))
         .mean("time").rename({"GRID_X_T": "xt_ocean", "GRID_Y_T": "yt_ocean", "ZT": "st_ocean"}))
woa13 = woa13.assign_coords(st_ocean=om2["T"].st_ocean)  # fix slightly different depth precision
woa13_zm = {v: atlantic_zonal_mean(woa13[name].where(om2[v].isel(time=0).notnull()), om2atl).load()
            for v, name in [("T", "temp"), ("S", "salt")]}

## Plot

Rows: ACCESS-OM3 − WOA23, ACCESS-OM2 − WOA13, WOA23 climatology. Colour limits follow Kiss et al. (2020) Fig. 12.

In [ ]:
panels = [  # (row data, title) for T and S
    ({v: om3_zm[v] - woa23_zm[v] for v in "TS"}, "ACCESS-OM3 − WOA23"),
    ({v: om2_zm[v] - woa13_zm[v] for v in "TS"}, "ACCESS-OM2 − WOA13"),
    (woa23_zm, "WOA23"),
]
style = {  # (variable, is_bias): (cmap, vmin, vmax, colourbar ticks, colourbar label)
    ("T", True): (cmocean.cm.balance, -1.8, 1.8, [-1.5, -0.75, 0, 0.75, 1.5], "Potential temperature bias (°C)"),
    ("S", True): (cmocean.cm.delta, -0.8, 0.8, [-0.8, -0.4, 0, 0.4, 0.8], "Salinity bias (psu)"),
    ("T", False): (cmocean.cm.thermal, 0, 25, [0, 5, 10, 15, 20, 25], "Potential temperature (°C)"),
    ("S", False): (cmocean.cm.haline, 33, 35.8, [33, 33.5, 34, 34.5, 35, 35.5], "Salinity (psu)"),
}

fig, axs = plt.subplots(3, 2, figsize=(11, 10), sharex=True, sharey=True, layout="constrained")
for row, (data, title) in enumerate(panels):
    is_bias = row < 2
    for col, v in enumerate("TS"):
        ax = axs[row, col]
        cmap, vmin, vmax, ticks, label = style[(v, is_bias)]
        da = data[v].rename({data[v].dims[0]: "depth", data[v].dims[1]: "lat"})
        levels = np.linspace(vmin, vmax, 37)
        p = ax.contourf(da.lat, da.depth, da, levels=levels, cmap=cmap, extend="both")
        ax.contour(da.lat, da.depth, da, levels=levels[::4], colors="k", linewidths=0.2)
        ax.set_facecolor("gray")
        ax.set_ylim(5500, 0)
        ax.set_title(f"({chr(ord('a') + 2 * row + col)}) {title}", fontsize=9)
        if row != 1:  # one colourbar per bias column (shared by rows 0-1) and per climatology panel
            fig.colorbar(p, ax=axs[:2, col] if is_bias else ax, label=label, ticks=ticks, shrink=0.8)
axs[2, 0].set_xlabel("Latitude (°N)")
axs[2, 1].set_xlabel("Latitude (°N)")
for ax in axs[:, 0]:
    ax.set_ylabel("Depth (m)")
fig.suptitle(f"Atlantic zonal mean, {period.start}–{period.stop}\nOM3: {om3exptname};  OM2: {om2exptname}", fontsize=10)

mkmd.savefig(fig, "Atlantic zonal mean temperature and salinity bias",
             f"Atlantic (34°S–65°N) zonal-mean potential temperature (left) and salinity (right), {period.start}–{period.stop} mean: "
             "ACCESS-OM3 minus WOA23, ACCESS-OM2 minus WOA13, and the WOA23 climatology. "
             "GitHub issue: [#195](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/195)", dpi=150)

In [ ]:
client.close()